# Day 199 — ResumeGapAI: Polish, README, and Deployment Prep
**Month 12 — Capstone Arc | Project 2 of 2 (Days 197-200)**

**Program position:** Day 199 of 203. Month 12 running total entering today: **565/565, 7/7 perfect days (Days 192-198)**.

**Today's scope (locked from Day 197):** Polish the ResumeGapAI Streamlit app into a client-credible, star-worthy deliverable - README with badges and demo, requirements.txt, deployment config, and a launch-ready checklist. **No new ML logic today.** `resumegap_engine.py` is locked (Section 1) - you do not touch it. Everything you build today wraps around it.

**Why this day exists:** A working model with no README, no screenshot, and no deploy link gets zero GitHub stars, because nobody clones a repo to find out if it works. The packaging *is* the product, from a portfolio-conversion standpoint. Day 200 (launch/star-seeding) only works if Day 199's artifacts are genuinely presentable.


## Section 1 - Raw Data (LOCKED - do not modify)

Two files are your immovable starting point today:

1. **`resumegap_engine.py`** - the finalized Day198 core engine (`clean_text`, `tfidf_similarity_naive`, `tfidf_similarity_fixed`, `gap_words`, `analyze_resume_gap`), scored 100/100 after the regex and stop-words fixes. Shown below for reference only.
2. **`app_skeleton.py`** - the bare Day198 UI: a title, two text areas, one button, one raw `st.write(result)`. Functionally correct, professionally unpresentable. This is what you polish.

Run the cell below to confirm the locked engine still behaves exactly as it did at Day198 sign-off, before building anything on top of it.


In [11]:
# Confirm the locked engine (unchanged from Day198 final submission)
from resumegap_engine import analyze_resume_gap
import inspect

resume_sample = "experienced data analyst skilled in sql python excel and tableau for reporting dashboards"
jd_sample = "seeking data analyst with strong sql python aws gcp kubernetes and machine learning experience for cloud data pipelines"

result = analyze_resume_gap(resume_sample, jd_sample)
print("Locked engine sanity check:")
print(result)


Locked engine sanity check:
{'similarity_naive': 24.28, 'similarity_fixed': 25.91, 'gap_words': ['experience', 'kubernetes', 'seeking', 'strong', 'aws', 'cloud', 'gcp', 'pipelines']}


**Raw Data note:** the printed `similarity_fixed`, `similarity_naive`, and `gap_words` values above are your engine's ground truth for today. Any screenshot, GIF, or README example you create should use *real* output from this cell (or the three locked Day198 resume/JD pairs) - never invented numbers. This is the same NRA discipline that applies to every other day: numbers come from `print()` output in the current run, not from memory.


## Section 2 - Concept Notes

### 2.1 What actually earns a GitHub star (the Day197 formula, applied)
Day197 scored ResumeGapAI on `demo_bucket` and `novelty_bucket` as two of five factors. Today is where those buckets get *earned*: a visitor decides to star a repo within about 10 seconds of landing on it, based on almost entirely non-code signals - a live demo link, one screenshot/GIF showing the tool actually working, and a README that states the problem in one sentence.

### 2.2 Anatomy of a star-worthy README
In order, top to bottom:
1. **Title + one-line pitch** - the problem, not the tech stack.
2. **Badges row** - Python version, framework, license, (optionally) live-demo status. Signals the repo is maintained, not abandoned.
3. **Screenshot or GIF** - proof it works, above the fold. This is the single highest-leverage element.
4. **Live demo link** - removes all friction; a visitor tries it before reading anything else.
5. **Quick start** (`pip install -r requirements.txt`, `streamlit run app.py`) - 3 lines max.
6. **How it works** - 2-3 sentences, plain language (naive vs corpus-corrected TF-IDF, gap words).
7. **Tech stack** - bullet list, not prose.
8. **License**.

### 2.3 Badges (shields.io)
Badges are just Markdown images pointing at a badge-generation service:
```markdown
![Python](https://img.shields.io/badge/python-3.10%2B-blue)
![Streamlit](https://img.shields.io/badge/streamlit-app-ff4b4b)
![License](https://img.shields.io/badge/license-MIT-green)
```
No API key, no account - fully free, which matters given the freelance/free-tools-first constraint.

### 2.4 requirements.txt - pin, don't float
`pandas` (unpinned) can silently pull a breaking major version months later when someone else clones your repo - the classic "works on my machine" failure, except the machine is a stranger's, evaluating whether to hire you. Pin exact versions you actually tested with: `pandas==2.2.2`, not `pandas`. This is the same discipline as `langchain==0.2.16` in the Month 10 stack, applied to a new project.

### 2.5 Streamlit Cloud vs Hugging Face Spaces (both free)
| | Streamlit Cloud | HF Spaces |
|---|---|---|
| Cost | Free (public repos) | Free (public repos) |
| Setup | Connect GitHub repo directly, auto-detects `app.py` | Push to a HF git remote, needs a `README.md` header block (`sdk: streamlit`) |
| Best for | Fastest path from GitHub repo to live URL | Slightly more ML-community visibility |
| Portfolio fit | Recommended default - GitHub-native, matches the `deepanshu0110` workflow | Optional second deploy for redundancy |

**Decision for today:** Streamlit Cloud, because it deploys straight from `Month12-Capstone-Portfolio` on `deepanshu0110` with zero extra accounts, matching the existing GitHub-first workflow.

### 2.6 Production error handling in Streamlit (new today, in scope)
Three failure modes any public Streamlit app hits within its first day of real traffic:
- **Empty input** -> crashes on `TfidfVectorizer.fit_transform([])` or produces a meaningless similarity score.
- **Uncaught exception** -> shows a raw Python traceback to a stranger, which reads as "unfinished project," not "in-progress error."
- **Silent rerun wipe** -> Streamlit reruns the whole script on any widget interaction; without `st.session_state`, results vanish the moment the user touches anything else.

Task 1, 3, and 4 below fix exactly these three, in that order.

### 2.7 .gitignore and .streamlit/config.toml
`.gitignore` keeps `__pycache__/`, `.ipynb_checkpoints/`, and local venv folders out of the repo - a repo littered with cache files reads as careless. `.streamlit/config.toml` lets you set a theme so the deployed app doesn't look like default Streamlit - a small visual signal that this was *finished*, not abandoned mid-build.


## Section 3 - Practice Guide

Complete these directly inside `app_skeleton.py` (rename it `app.py` when done), plus the supporting files listed. Each task's TODOs are already marked as comments in `app_skeleton.py` - this notebook only tracks scope and constraints, the actual code goes in the .py file.

**Constraint reminder:** everything here uses only `streamlit`, `pandas` (if needed), and Python standard library - no new packages beyond what Day198 already introduced (`scikit-learn`). No forward references to LangChain/agents (that's post-Month-12 scope).


### Task 1 - Input validation & error handling (20 pts)
In `app_skeleton.py`: complete TODO 1a, 1b, 1c.
Test it by running the app and deliberately (a) clicking Analyze with both fields empty, (b) pasting a 3-word resume, (c) temporarily breaking something to confirm the try/except doesn't leak a traceback.


### Task 2 - UX polish: spinner, layout, metrics (25 pts)
Complete TODO 2a, 2b, 2c. Run the app locally (`streamlit run app_skeleton.py`) and describe your layout decision below - it becomes part of your README later.


In [12]:
task2_layout_note = "I used two side‑by‑side columns with st.metric() to display the naive and corrected scores. This places them at the top for immediate visibility, and the caption below explains why both are shown (transparency vs. trust). The gap words are listed underneath as a bulleted list, which is scannable and actionable."
print(task2_layout_note)

I used two side‑by‑side columns with st.metric() to display the naive and corrected scores. This places them at the top for immediate visibility, and the caption below explains why both are shown (transparency vs. trust). The gap words are listed underneath as a bulleted list, which is scannable and actionable.


### Task 3 - Downloadable report (20 pts)
Complete TODO 3a, 3b in `app_skeleton.py`. Below, build and print the SAME report string logic against the real `result` from Section 1, so you can verify the format before wiring it into Streamlit.


In [13]:
def build_report(result: dict, band: str) -> str:
    lines = []
    lines.append("=== ResumeGapAI Report ===\n")
    lines.append(f"Corpus‑Corrected Similarity: {result['similarity_fixed']:.1f}%")
    lines.append(f"Naive TF‑IDF Similarity:     {result['similarity_naive']:.1f}%")
    lines.append(f"Match Band:                  {band}")
    lines.append("\nTop skill gaps to address:")
    if result['gap_words']:
        for w in result['gap_words']:
            lines.append(f"  - {w}")
    else:
        lines.append("  None – excellent alignment!")
    return "\n".join(lines)

# Test using the result from Section 1
# (In the notebook, result is already defined from the engine sanity check)
report_text = build_report(result, "Moderate Match")  # compute band appropriately
print(report_text)

=== ResumeGapAI Report ===

Corpus‑Corrected Similarity: 25.9%
Naive TF‑IDF Similarity:     24.3%
Match Band:                  Moderate Match

Top skill gaps to address:
  - experience
  - kubernetes
  - seeking
  - strong
  - aws
  - cloud
  - gcp
  - pipelines


### Task 4 - Session state (15 pts)
Complete TODO 4a, 4b in `app_skeleton.py`. No notebook code needed - session state doesn't exist outside a running Streamlit app. Note in one sentence why this matters for THIS app specifically.


In [14]:
task4_note = "Session state is crucial here because Streamlit reruns the entire script on every interaction; without it, the analysis result would disappear as soon as the user resizes the text area or toggles any widget, breaking the user's workflow."
print(task4_note)

Session state is crucial here because Streamlit reruns the entire script on every interaction; without it, the analysis result would disappear as soon as the user resizes the text area or toggles any widget, breaking the user's workflow.


### Task 5 - README.md, requirements.txt, deployment (20 pts)
Build three files (as separate files, not notebook cells): `README.md`, `requirements.txt`, `.streamlit/config.toml`. Follow the badge/section structure from Concept Notes 2.2-2.3. Then answer the NRA deployment decision below with real reasoning (restate Concept Notes 2.5 in your own words, as if explaining to a client).


In [ ]:
deployment_decision = {
    "number": "$0 – both platforms are free for public repos",
    "reason": "Streamlit Cloud connects directly to my GitHub repository (deepanshu0110/ResumeGapAI) and automatically deploys from the main branch when I push changes – no separate account or manual upload is required. The app is detected by Streamlit because the entry file is named `app.py`, which is the default filename the platform looks for.",
    "action": "I will connect deepanshu0110/ResumeGapAI on Streamlit Cloud, obtain the live URL, and add it as a badge in the README and in the app sidebar before Day 200 launch."
}

{'number': '$0 – both platforms are free for public repos', 'reason': 'Streamlit Cloud connects directly to my GitHub repository (deepanshu0110/ResumeGapAI) and automatically deploys from the main branch when I push changes – no separate account or manual upload is required. The app is detected by Streamlit because the entry file is named `app.py`, which is the default filename the platform looks for.', 'action': 'I will connect the repo on Streamlit Cloud, obtain the live URL, and add it as a badge in the README and in the app sidebar before Day 200 launch.'}


## Section 4 - Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 - Input validation & error handling | 20 | Empty-field guard, min-length guard, try/except with no raw traceback exposure |
| Task 2 - UX polish (spinner, layout, metrics) | 25 | Spinner present, `st.columns` + `st.metric` used correctly, caption explains dual-score design |
| Task 3 - Downloadable report | 20 | Report string has all 3 pieces of data, `st.download_button` wired with correct `mime` |
| Task 4 - Session state | 15 | Result persists across unrelated reruns, correctly initialized before first use |
| Task 5 - README + requirements.txt + deployment NRA | 20 | Badges present, screenshot/GIF placeholder, quick start, correct pinned versions, NRA has no Reason/Action bleed |
| **Total** | **100** | Plus 10 star bonus if all 5 tasks pass with zero resubmits |

**NRA discipline reminder for Task 5:** Reason must explain *why Streamlit Cloud fits this specific setup* (mechanism), not restate what the Action will do or what Streamlit Cloud is generally "good at" (benefit language). This is the same Reason-to-Action bleed pattern flagged on Days 192, 194, and 197 - standing self-check applies here too, even though today's task is packaging, not modeling.


## Interview-Framing Answer

*"How would you explain what you built today if a client or interviewer asked?"*

> "After the core matching engine worked, I spent a day specifically on production-readiness - the kind of thing that separates a notebook experiment from something a client can actually use. That meant input validation so it fails gracefully instead of crashing on bad input, a downloadable report so the output is actually usable outside the browser tab, and a README with a live demo link so anyone evaluating my work can try it in under ten seconds without reading code. I chose Streamlit Cloud for deployment specifically because it connects directly to my existing GitHub repo with no extra setup - for a freelance portfolio, minimizing friction between 'here's my GitHub' and 'here's it running' is the whole point."

## Key Takeaway

**A working model and a star-worthy, client-ready deliverable are two different projects - and the second one is almost entirely non-ML work (error handling, README, one honest screenshot, a free deploy link).** The gap between "it works on my machine" and "a stranger trusts it in ten seconds" is closed by exactly the five things practiced today, not by more modeling.
